## Data Collection Engine Test

Day 3 작업 중 데이터 수집 모듈 기능 테스트

테스트 대상:
- SearchClient (Serper & Brave Search API)
- ArxivCollector
- NewsCollector

In [1]:
import asyncio
import json
import sys
from pathlib import Path

from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment variables
load_dotenv(project_root / ".env")

from app.collectors.arxiv import ArxivCollector, get_popular_ai_categories
from app.collectors.base import CollectedData, SourceType
from app.collectors.news import NewsCollector, get_ai_news_domains
from app.collectors.search_client import SearchClient

print("✓ Setup complete")

✓ Setup complete


### 1. SearchClient - Serper API Test

Serper Search API를 사용한 일반 검색 테스트

In [2]:
# Serper 일반 검색
search_client = SearchClient()

query = "GPT-4 architecture"
results = await search_client.serper_search(
    query=query,
    num_results=5,
    search_type="search"
)

print(f"Serper Search Results for '{query}':")
print(f"Total results: {len(results)}\n")

for i, result in enumerate(results, 1):
    print(f"{i}. {result['title']}")
    print(f"   URL: {result['link']}")
    print(f"   Snippet: {result['snippet'][:100]}...")
    print()

Serper Search Results for 'GPT-4 architecture':
Total results: 5

1. What's new in GPT-4: Architecture and Capabilities
   URL: https://medium.com/@amol-wagh/whats-new-in-gpt-4-an-overview-of-the-gpt-4-architecture-and-capabilities-of-next-generation-ai-900c445d5ffe
   Snippet: GPT-4 is a new language model created by OpenAI that is a large multimodal that can accept image and...

2. OpenAI GPT-4: Architecture, Interfaces, Pricing & Alternatives
   URL: https://obot.ai/resources/learning-center/openai/
   Snippet: The architecture of GPT-4 marks a significant departure from previous models by adopting a mixture o...

3. GPT-4
   URL: https://openai.com/index/gpt-4-research/
   Snippet: GPT-4 is a large multimodal model (accepting image and text inputs, emitting text outputs) that, whi...

4. GPT-4 Architecture, Infrastructure, Training Dataset, Costs ...
   URL: https://newsletter.semianalysis.com/p/gpt-4-architecture-infrastructure
   Snippet: Model Architecture. GPT-4 is more than 10

### 2. SearchClient - Serper News Search

최신 뉴스 검색 및 날짜 필터링 테스트

In [3]:
# Serper 뉴스 검색 (최근 1주일)
news_query = "artificial intelligence"
news_results = await search_client.serper_search(
    query=news_query,
    num_results=5,
    search_type="news",
    date_filter="w"  # w = week, d = day, m = month
)

print(f"Serper News Results for '{news_query}' (past week):")
print(f"Total results: {len(news_results)}\n")

for i, result in enumerate(news_results, 1):
    print(f"{i}. {result['title']}")
    print(f"   Source: {result.get('source', 'N/A')}")
    print(f"   Date: {result.get('date', 'N/A')}")
    print(f"   URL: {result['link']}")
    print()

Serper News Results for 'artificial intelligence' (past week):
Total results: 10

1. Artificial intelligence (AI) | Definition, Examples, Types, Applications, Companies, & Facts
   Source: Britannica
   Date: 2 days ago
   URL: https://www.britannica.com/technology/artificial-intelligence

2. HHS Releases Strategy Positioning Artificial Intelligence as the Core of Health Innovation | Insights
   Source: Holland & Knight
   Date: 3 days ago
   URL: https://www.hklaw.com/en/insights/publications/2025/12/hhs-releases-strategy-positioning-artificial-intelligence

3. President Signs Executive Order to Limit State Regulation of Artificial Intelligence
   Source: Littler Mendelson P.C.
   Date: 2 days ago
   URL: https://www.littler.com/news-analysis/asap/president-signs-executive-order-limit-state-regulation-artificial-intelligence

4. Artificial intelligence agents for biology
   Source: Nature
   Date: 5 days ago
   URL: https://www.nature.com/articles/s41592-025-02958-y

5. Financial Serv

### 3. SearchClient - Serper Scholar Search

학술 논문 검색 테스트

In [4]:
# Serper Scholar 검색
scholar_query = "transformer neural network"
scholar_results = await search_client.serper_search(
    query=scholar_query,
    num_results=5,
    search_type="scholar"
)

print(f"Serper Scholar Results for '{scholar_query}':")
print(f"Total results: {len(scholar_results)}\n")

for i, result in enumerate(scholar_results, 1):
    print(f"{i}. {result['title']}")
    print(f"   Publication: {result.get('publication', 'N/A')}")
    print(f"   Year: {result.get('year', 'N/A')}")
    print(f"   Cited by: {result.get('cited_by', 'N/A')}")
    print(f"   URL: {result['link']}")
    print()

Serper Scholar Results for 'transformer neural network':
Total results: 10

1. Transformer in convolutional neural networks
   Publication: None
   Year: 2021
   Cited by: None
   URL: https://homes.esat.kuleuven.be/~konijn/publications/2021/Liu2.pdf

2. R-transformer: Recurrent neural network enhanced transformer
   Publication: None
   Year: 1907
   Cited by: None
   URL: https://arxiv.org/abs/1907.05572

3. Transforming the language of life: transformer neural networks for protein prediction tasks
   Publication: None
   Year: 2020
   Cited by: None
   URL: https://dl.acm.org/doi/abs/10.1145/3388440.3412467

4. A comparison of transformer and recurrent neural networks on multilingual neural machine translation
   Publication: None
   Year: 2018
   Cited by: None
   URL: https://aclanthology.org/C18-1054/

5. Transformer neural networks for interpretable flood forecasting
   Publication: None
   Year: 2023
   Cited by: None
   URL: https://www.sciencedirect.com/science/article/pii/S1

### 4. SearchClient - Brave Search Test

Brave Search API를 사용한 검색 테스트 (Serper API가 없는 경우 대체)

In [5]:
# Brave 일반 검색
try:
    brave_query = "machine learning trends 2024"
    brave_results = await search_client.brave_search(
        query=brave_query,
        num_results=5,
        search_type="web",
        freshness="pw"  # pw = past week, pd = past day, pm = past month
    )
    
    print(f"Brave Search Results for '{brave_query}':")
    print(f"Total results: {len(brave_results)}\n")
    
    for i, result in enumerate(brave_results, 1):
        print(f"{i}. {result['title']}")
        print(f"   Source: {result.get('source', 'N/A')}")
        print(f"   URL: {result['link']}")
        print(f"   Snippet: {result['snippet'][:100]}...")
        print()
except Exception as e:
    print(f"Brave API not available: {e}")

Brave Search Results for 'machine learning trends 2024':
Total results: 5

1. Machine Learning Statistics 2025: Market Growth, Adoption, ROI, Jobs, and Future Trends
   Source: MindInventory
   URL: https://www.mindinventory.com/blog/machine-learning-statistics/
   Snippet: Businesses are seeing a reduction in difficulties in hiring machine learning engineers, a drop from ...

2. AI's Impact on Graduate Jobs: A 2025 Data Analysis | IntuitionLabs
   Source: IntuitionLabs
   URL: https://intuitionlabs.ai/articles/ai-impact-graduate-jobs-2025
   Snippet: Shift in Career Choices: Early evidence suggests some students are adjusting their education plans. ...

3. AI Trends Report 2024 AI’s Growing Role in Software Development
   Source: baike know
   URL: https://duoduono1.com/blog/ai-trends-report-2024/
   Snippet: The uptake of AI tools such as ChatGPT, GitHub Copilot, and Bard among developers is a testament to ...

4. NeurIPS 2025: A Guide to Key Papers, Trends & Stats | IntuitionLabs
  

### 5. SearchClient - Unified Search Interface

통합 검색 인터페이스 테스트

In [6]:
# 통합 검색 인터페이스
unified_query = "deep learning"

# Serper를 기본으로 사용
unified_results = await search_client.search(
    query=unified_query,
    num_results=3,
    provider="serper",
    search_type="search"
)

print(f"Unified Search Results (using Serper):")
print(f"Query: '{unified_query}'")
print(f"Total results: {len(unified_results)}\n")

for i, result in enumerate(unified_results, 1):
    print(f"{i}. {result['title']}")
    print(f"   {result['link']}")
    print()

Unified Search Results (using Serper):
Query: 'deep learning'
Total results: 3

1. Deep learning
   https://en.wikipedia.org/wiki/Deep_learning

2. What Is Deep Learning? | IBM
   https://www.ibm.com/think/topics/deep-learning

3. DeepLearning.AI: Start or Advance Your Career in AI
   https://www.deeplearning.ai/



### 6. ArxivCollector - Basic Paper Collection

arXiv에서 논문 수집 기본 테스트

In [7]:
# arXiv 기본 검색
arxiv_collector = ArxivCollector()

papers = await arxiv_collector.collect(
    query="large language model",
    limit=5
)

print(f"ArXiv Papers Collected: {len(papers)}\n")

for i, paper in enumerate(papers, 1):
    print(f"{i}. {paper.title}")
    print(f"   Source: {paper.source_name}")
    print(f"   Type: {paper.source_type.value}")
    print(f"   URL: {paper.url}")
    print(f"   Authors: {', '.join(paper.metadata['authors'][:3])}...")
    print(f"   Category: {paper.metadata['primary_category']}")
    print(f"   Published: {paper.metadata['published'][:10]}")
    print(f"   Abstract preview: {paper.content[:150]}...")
    print()

ArXiv Papers Collected: 5

1. Learning From Failure: Integrating Negative Examples when Fine-tuning Large Language Models as Agents
   Source: arXiv
   Type: paper
   URL: http://arxiv.org/abs/2402.11651v2
   Authors: Renxi Wang, Haonan Li, Xudong Han...
   Category: cs.CL
   Published: 2024-02-18
   Abstract preview: Large language models (LLMs) have achieved success in acting as agents, which interact with environments through tools such as search engines. However...

2. Demystifying Instruction Mixing for Fine-tuning Large Language Models
   Source: arXiv
   Type: paper
   URL: http://arxiv.org/abs/2312.10793v3
   Authors: Renxi Wang, Haonan Li, Minghao Wu...
   Category: cs.CL
   Published: 2023-12-17
   Abstract preview: Instruction tuning significantly enhances the performance of large language models (LLMs) across various tasks. However, the procedure to optimizing t...

3. WizardLM: Empowering large pre-trained language models to follow complex instructions
   Source: arXiv
   

### 7. ArxivCollector - Category Filtering

특정 AI 카테고리만 필터링하여 수집

In [8]:
# AI 관련 카테고리만 필터링
ai_categories = get_popular_ai_categories()
print(f"Popular AI Categories: {ai_categories}\n")

filtered_papers = await arxiv_collector.collect(
    query="attention mechanism",
    limit=5,
    filters={
        "categories": ["cs.AI", "cs.LG"],  # AI와 Machine Learning만
        "sort_by": "relevance",
        "sort_order": "descending"
    }
)

print(f"Filtered ArXiv Papers (cs.AI, cs.LG only): {len(filtered_papers)}\n")

for i, paper in enumerate(filtered_papers, 1):
    print(f"{i}. {paper.title}")
    print(f"   Categories: {', '.join(paper.metadata['categories'])}")
    print(f"   Primary: {paper.metadata['primary_category']}")
    print(f"   arXiv ID: {paper.metadata['arxiv_id']}")
    print(f"   PDF: {paper.metadata['pdf_url']}")
    print()

Popular AI Categories: ['cs.AI', 'cs.LG', 'cs.CL', 'cs.CV', 'cs.NE', 'cs.RO', 'stat.ML']

Filtered ArXiv Papers (cs.AI, cs.LG only): 5

1. Déjà vu: A Contextualized Temporal Attention Mechanism for Sequential Recommendation
   Categories: cs.IR, cs.CL, cs.LG
   Primary: cs.IR
   arXiv ID: 2002.00741v1
   PDF: https://arxiv.org/pdf/2002.00741v1

2. Pay Attention to What You Need
   Categories: cs.CL, cs.AI
   Primary: cs.CL
   arXiv ID: 2307.13365v3
   PDF: https://arxiv.org/pdf/2307.13365v3

3. Benign Overfitting in Token Selection of Attention Mechanism
   Categories: cs.LG
   Primary: cs.LG
   arXiv ID: 2409.17625v3
   PDF: https://arxiv.org/pdf/2409.17625v3

4. Efficient Attention via Control Variates
   Categories: cs.LG, cs.CL, cs.CV
   Primary: cs.LG
   arXiv ID: 2302.04542v1
   PDF: https://arxiv.org/pdf/2302.04542v1

5. Learning Efficient Algorithms with Hierarchical Attentive Memory
   Categories: cs.LG
   Primary: cs.LG
   arXiv ID: 1602.03218v2
   PDF: https://arxiv.org/pdf/

### 8. ArxivCollector - Sort Options Test

정렬 옵션 테스트 (최신순, 관련도순 등)

In [9]:
# 최신순 정렬
recent_papers = await arxiv_collector.collect(
    query="GPT",
    limit=3,
    filters={
        "sort_by": "submitted",  # submitted, last_updated, relevance
        "sort_order": "descending"
    }
)

print(f"Recent GPT Papers (sorted by submission date):")
print(f"Total: {len(recent_papers)}\n")

for i, paper in enumerate(recent_papers, 1):
    print(f"{i}. {paper.title}")
    print(f"   Published: {paper.metadata['published'][:10]}")
    print(f"   Updated: {paper.metadata['updated'][:10]}")
    print()

Recent GPT Papers (sorted by submission date):
Total: 3

1. BabyVLM-V2: Toward Developmentally Grounded Pretraining and Benchmarking of Vision Foundation Models
   Published: 2025-12-11
   Updated: 2025-12-11

2. SparseSwaps: Tractable LLM Pruning Mask Refinement at Scale
   Published: 2025-12-11
   Updated: 2025-12-11

3. LabelFusion: Learning to Fuse LLMs and Transformer Classifiers for Robust Text Classification
   Published: 2025-12-11
   Updated: 2025-12-11



### 9. ArxivCollector - CollectedData Structure Test

수집된 데이터 구조 검증

In [10]:
# 단일 논문으로 데이터 구조 확인
single_paper = await arxiv_collector.collect(
    query="transformer",
    limit=1
)

if single_paper:
    paper = single_paper[0]
    
    print("CollectedData Structure Test:")
    print("=" * 60)
    print(f"Title: {paper.title}")
    print(f"Content type: {type(paper.content).__name__}")
    print(f"Content length: {len(paper.content)} chars")
    print(f"URL: {paper.url}")
    print(f"Source type: {paper.source_type} (Enum: {isinstance(paper.source_type, SourceType)})")
    print(f"Source name: {paper.source_name}")
    print(f"Collected at: {paper.collected_at}")
    print(f"\nMetadata keys: {list(paper.metadata.keys())}")
    print(f"\nMetadata content:")
    print(json.dumps(paper.metadata, indent=2, default=str))
    
    # to_dict() 메서드 테스트
    print("\n" + "=" * 60)
    print("to_dict() conversion test:")
    paper_dict = paper.to_dict()
    print(f"Dict keys: {list(paper_dict.keys())}")
    print(f"source_type as string: {paper_dict['source_type']}")
    print(f"collected_at as ISO: {paper_dict['collected_at']}")

CollectedData Structure Test:
Title: PyramidTNT: Improved Transformer-in-Transformer Baselines with Pyramid Architecture
Content type: str
Content length: 781 chars
URL: http://arxiv.org/abs/2201.00978v1
Source type: SourceType.PAPER (Enum: True)
Source name: arXiv
Collected at: 2025-12-14 12:30:39.116994

Metadata keys: ['arxiv_id', 'authors', 'primary_category', 'categories', 'published', 'updated', 'pdf_url', 'comment', 'journal_ref', 'doi']

Metadata content:
{
  "arxiv_id": "2201.00978v1",
  "authors": [
    "Kai Han",
    "Jianyuan Guo",
    "Yehui Tang",
    "Yunhe Wang"
  ],
  "primary_category": "cs.CV",
  "categories": [
    "cs.CV"
  ],
  "published": "2022-01-04T04:56:57+00:00",
  "updated": "2022-01-04T04:56:57+00:00",
  "pdf_url": "https://arxiv.org/pdf/2201.00978v1",
  "comment": "Tech Report. An extension of \"Transformer in Transformer\" (arXiv:2103.00112)",
  "journal_ref": null,
  "doi": null
}

to_dict() conversion test:
Dict keys: ['title', 'content', 'url', 'sourc

### 10. NewsCollector - Basic News Collection

뉴스 수집 기본 테스트

In [11]:
# 기본 뉴스 수집 (기본 도메인 사용)
news_collector = NewsCollector(search_provider="serper")

articles = await news_collector.collect(
    query="artificial intelligence",
    limit=5
)

print(f"News Articles Collected: {len(articles)}\n")

for i, article in enumerate(articles, 1):
    print(f"{i}. {article.title}")
    print(f"   Source: {article.source_name}")
    print(f"   Type: {article.source_type.value}")
    print(f"   URL: {article.url}")
    print(f"   Published: {article.metadata.get('published_date', 'N/A')}")
    print(f"   Snippet: {article.content[:100]}...")
    print()

News Articles Collected: 10

1. Is language the same as intelligence? The AI industry desperately needs it to be
   Source: News
   Type: news
   URL: https://www.theverge.com/ai-artificial-intelligence/827820/large-language-models-ai-intelligence-neuroscience-problems
   Published: 3 weeks ago
   Snippet: Neuroscience indicates language is distinct from thought, raising questions about whether AI large l...

2. OpenAI, Anthropic, and Block Are Teaming Up to Make AI Agents Play Nice
   Source: News
   Type: news
   URL: https://www.wired.com/story/openai-anthropic-and-block-are-teaming-up-on-ai-agent-standards/
   Published: 4 days ago
   Snippet: OpenAI, Anthropic, and Block have cofounded a new open source organization—the Agentic AI Foundation...

3. Trump Signs Executive Order That Threatens to Punish States for Passing AI Laws
   Source: News
   Type: news
   URL: https://www.wired.com/story/trump-signs-executive-order-ai-state-laws/
   Published: 2 days ago
   Snippet: The order 

### 11. NewsCollector - Domain Filtering

특정 도메인만 필터링하여 뉴스 수집

In [12]:
# 기본 뉴스 도메인 확인
default_domains = get_ai_news_domains()
print(f"Default AI News Domains:")
for domain in default_domains:
    print(f"  - {domain}")
print()

# 특정 도메인만 선택
filtered_articles = await news_collector.collect(
    query="ChatGPT",
    limit=5,
    filters={
        "domains": ["techcrunch.com", "venturebeat.com"],
        "date_filter": "w"  # 최근 1주일
    }
)

print(f"Filtered News (TechCrunch & VentureBeat only): {len(filtered_articles)}\n")

for i, article in enumerate(filtered_articles, 1):
    print(f"{i}. {article.title}")
    print(f"   Source: {article.metadata.get('source_name', 'N/A')}")
    print(f"   URL: {article.url}")
    print()

Default AI News Domains:
  - techcrunch.com
  - venturebeat.com
  - technologyreview.com
  - theverge.com
  - wired.com
  - arstechnica.com
  - zdnet.com

Filtered News (TechCrunch & VentureBeat only): 10

1. OpenAI fires back at Google with GPT-5.2 after ‘code red’ memo
   Source: TechCrunch
   URL: https://techcrunch.com/2025/12/11/openai-fires-back-at-google-with-gpt-5-2-after-code-red-memo/

2. GPT-5.2 first impressions: a powerful update, especially for business tasks and workflows
   Source: VentureBeat
   URL: https://venturebeat.com/ai/gpt-5-2-first-impressions-a-powerful-update-especially-for-business-tasks

3. OpenAI's GPT-5.2 is here: what enterprises need to know
   Source: VentureBeat
   URL: https://venturebeat.com/ai/openais-gpt-5-2-is-here-what-enterprises-need-to-know

4. Google launches sub-$5 AI Plus plan in India to compete with ChatGPT Go
   Source: TechCrunch
   URL: https://techcrunch.com/2025/12/10/google-launches-sub-5-ai-plus-plan-in-india-to-compete-with-chat

### 12. NewsCollector - Query Building Test

도메인 쿼리 생성 로직 테스트

In [13]:
# 도메인 쿼리 빌드 테스트
test_query = "GPT-4"
test_domains = ["techcrunch.com", "wired.com", "theverge.com"]

domain_query = news_collector._build_domain_query(test_query, test_domains)

print("Domain Query Building Test:")
print(f"Original query: {test_query}")
print(f"Domains: {test_domains}")
print(f"\nGenerated query:")
print(f"  {domain_query}")
print(f"\nExpected format: 'query (site:domain1 OR site:domain2 OR ...)'")

Domain Query Building Test:
Original query: GPT-4
Domains: ['techcrunch.com', 'wired.com', 'theverge.com']

Generated query:
  GPT-4 (site:techcrunch.com OR site:wired.com OR site:theverge.com)

Expected format: 'query (site:domain1 OR site:domain2 OR ...)'


### 13. Batch Collection Test

여러 Collector를 동시에 실행하여 데이터 수집

In [14]:
# 여러 소스에서 동시에 수집
async def collect_from_all_sources(query: str, limit: int = 3):
    """모든 Collector에서 동시에 데이터 수집"""
    arxiv = ArxivCollector()
    news = NewsCollector()
    
    # 비동기로 동시 실행
    papers, articles = await asyncio.gather(
        arxiv.collect(query, limit),
        news.collect(query, limit),
        return_exceptions=True
    )
    
    return {
        "papers": papers if not isinstance(papers, Exception) else [],
        "articles": articles if not isinstance(articles, Exception) else [],
    }

# 테스트 실행
batch_query = "transformer"
results = await collect_from_all_sources(batch_query, limit=3)

print(f"Batch Collection Results for '{batch_query}':")
print(f"Papers collected: {len(results['papers'])}")
print(f"Articles collected: {len(results['articles'])}")
print(f"\nTotal items: {len(results['papers']) + len(results['articles'])}")

print("\n" + "=" * 60)
print("Papers:")
for i, paper in enumerate(results['papers'], 1):
    print(f"{i}. {paper.title[:60]}...")
    print(f"   [{paper.source_type.value}] {paper.url}")

print("\n" + "=" * 60)
print("Articles:")
for i, article in enumerate(results['articles'], 1):
    print(f"{i}. {article.title[:60]}...")
    print(f"   [{article.source_type.value}] {article.url}")

Batch Collection Results for 'transformer':
Papers collected: 3
Articles collected: 10

Total items: 13

Papers:
1. PyramidTNT: Improved Transformer-in-Transformer Baselines wi...
   [paper] http://arxiv.org/abs/2201.00978v1
2. Learning to Cluster Faces via Transformer...
   [paper] http://arxiv.org/abs/2104.11502v1
3. MLP Can Be A Good Transformer Learner...
   [paper] http://arxiv.org/abs/2404.05657v1

Articles:
1. Large language models can do jaw-dropping things. But nobody...
   [news] https://www.technologyreview.com/2024/03/04/1089403/large-language-models-amazing-but-nobody-knows-why/
2. Mistral launches powerful Devstral 2 coding model including ...
   [news] https://venturebeat.com/ai/mistral-launches-powerful-devstral-2-coding-model-including-open-source
3. Inside Rivian’s big bet on AI-powered self-driving...
   [news] https://techcrunch.com/2025/12/12/inside-rivians-big-et-on-ai-powered-self-driving/
4. Riding onboard with Rivian’s race to autonomy...
   [news] https://tech

### 14. Error Handling Test

에러 상황 처리 테스트

In [15]:
# 빈 쿼리 테스트
try:
    empty_results = await arxiv_collector.collect(query="", limit=1)
    print(f"Empty query result: {len(empty_results)} items")
except Exception as e:
    print(f"✓ Empty query error handled: {type(e).__name__}")

# 매우 큰 limit 테스트
try:
    large_results = await arxiv_collector.collect(query="AI", limit=1000)
    print(f"\n✓ Large limit handled: {len(large_results)} items (max enforced by API)")
except Exception as e:
    print(f"Large limit error: {type(e).__name__}: {e}")

# 잘못된 카테고리 필터
try:
    invalid_cat_results = await arxiv_collector.collect(
        query="AI",
        limit=1,
        filters={"categories": ["invalid.category"]}
    )
    print(f"\nInvalid category result: {len(invalid_cat_results)} items (API may ignore)")
except Exception as e:
    print(f"✓ Invalid category error handled: {type(e).__name__}")

# 잘못된 검색 프로바이더
try:
    invalid_search = await search_client.search(
        query="test",
        provider="invalid_provider"
    )
except ValueError as e:
    print(f"\n✓ Invalid provider error handled: {e}")

ArxivCollector error: Page request resulted in HTTP 400 (https://export.arxiv.org/api/query?search_query=&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100)
Attempt 1/3 failed for collect: Failed to collect from arXiv: Page request resulted in HTTP 400 (https://export.arxiv.org/api/query?search_query=&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100). Retrying in 1.00s...
ArxivCollector error: Page request resulted in HTTP 400 (https://export.arxiv.org/api/query?search_query=&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100)
Attempt 2/3 failed for collect: Failed to collect from arXiv: Page request resulted in HTTP 400 (https://export.arxiv.org/api/query?search_query=&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100). Retrying in 2.00s...
ArxivCollector error: Page request resulted in HTTP 400 (https://export.arxiv.org/api/query?search_query=&id_list=&sortBy=relevance&sortOrder=descending&start=0&max

✓ Empty query error handled: CollectorError

✓ Large limit handled: 1000 items (max enforced by API)

Invalid category result: 0 items (API may ignore)

✓ Invalid provider error handled: Invalid search provider: invalid_provider


### 15. Performance Benchmark

각 Collector의 성능 측정

In [16]:
import time

async def benchmark_collector(collector, name, query, limit):
    """Collector 성능 측정"""
    start = time.time()
    results = await collector.collect(query, limit)
    elapsed = time.time() - start
    return {
        "name": name,
        "query": query,
        "limit": limit,
        "collected": len(results),
        "time": elapsed,
        "avg_per_item": elapsed / len(results) if results else 0
    }

In [17]:
# 벤치마크 실행
benchmark_query = "deep learning"
benchmark_limit = 5

arxiv_bench = await benchmark_collector(
    ArxivCollector(), 
    "ArxivCollector", 
    benchmark_query, 
    benchmark_limit
)

news_bench = await benchmark_collector(
    NewsCollector(), 
    "NewsCollector", 
    benchmark_query, 
    benchmark_limit
)

print("Performance Benchmark Results:")
print("=" * 60)
for bench in [arxiv_bench, news_bench]:
    print(f"\n{bench['name']}:")
    print(f"  Query: '{bench['query']}'")
    print(f"  Requested: {bench['limit']} items")
    print(f"  Collected: {bench['collected']} items")
    print(f"  Total time: {bench['time']:.2f}s")
    print(f"  Avg per item: {bench['avg_per_item']:.2f}s")

Performance Benchmark Results:

ArxivCollector:
  Query: 'deep learning'
  Requested: 5 items
  Collected: 5 items
  Total time: 1.75s
  Avg per item: 0.35s

NewsCollector:
  Query: 'deep learning'
  Requested: 5 items
  Collected: 10 items
  Total time: 1.54s
  Avg per item: 0.15s


### 16. Data Quality Check

수집된 데이터의 품질 검증

In [18]:
def check_data_quality(collected_items: list[CollectedData]) -> dict:
    """수집된 데이터 품질 체크"""
    if not collected_items:
        return {"status": "empty", "errors": ["No items collected"]}
    
    errors = []
    warnings = []
    
    for i, item in enumerate(collected_items):
        # 필수 필드 체크
        if not item.title or not item.title.strip():
            errors.append(f"Item {i}: Missing or empty title")
        
        if not item.content or not item.content.strip():
            errors.append(f"Item {i}: Missing or empty content")
        
        if not item.url or not item.url.startswith("http"):
            errors.append(f"Item {i}: Invalid URL: {item.url}")
        
        # 경고 체크
        if len(item.content) < 50:
            warnings.append(f"Item {i}: Very short content ({len(item.content)} chars)")
        
        if not item.metadata:
            warnings.append(f"Item {i}: No metadata")
    
    return {
        "status": "error" if errors else "warning" if warnings else "ok",
        "total_items": len(collected_items),
        "errors": errors,
        "warnings": warnings,
        "avg_title_length": sum(len(item.title) for item in collected_items) / len(collected_items),
        "avg_content_length": sum(len(item.content) for item in collected_items) / len(collected_items),
    }

In [19]:
# 품질 체크 실행
test_papers = await arxiv_collector.collect(query="neural network", limit=5)
quality_report = check_data_quality(test_papers)

print("Data Quality Report:")
print("=" * 60)
print(f"Status: {quality_report['status'].upper()}")
print(f"Total items: {quality_report['total_items']}")
print(f"Avg title length: {quality_report['avg_title_length']:.1f} chars")
print(f"Avg content length: {quality_report['avg_content_length']:.1f} chars")

if quality_report['errors']:
    print(f"\n❌ Errors ({len(quality_report['errors'])}):")
    for error in quality_report['errors']:
        print(f"  - {error}")

if quality_report['warnings']:
    print(f"\n⚠️  Warnings ({len(quality_report['warnings'])}):")
    for warning in quality_report['warnings'][:3]:  # Show first 3
        print(f"  - {warning}")

if quality_report['status'] == 'ok':
    print("\n✅ All quality checks passed!")

Data Quality Report:
Status: OK
Total items: 5
Avg title length: 85.4 chars
Avg content length: 1199.0 chars

✅ All quality checks passed!


### Summary

이 노트북에서 다룬 내용:

#### SearchClient Tests (1-5)
1. ✓ Serper 일반 검색
2. ✓ Serper 뉴스 검색 (날짜 필터)
3. ✓ Serper Scholar 검색
4. ✓ Brave Search 검색
5. ✓ 통합 검색 인터페이스

#### ArxivCollector Tests (6-9)
6. ✓ 기본 논문 수집
7. ✓ 카테고리 필터링
8. ✓ 정렬 옵션 (최신순, 관련도순)
9. ✓ CollectedData 구조 검증

#### NewsCollector Tests (10-12)
10. ✓ 기본 뉴스 수집
11. ✓ 도메인 필터링
12. ✓ 쿼리 생성 로직 검증

#### Integration Tests (13-16)
13. ✓ 배치 수집 (여러 소스 동시)
14. ✓ 에러 핸들링
15. ✓ 성능 벤치마크
16. ✓ 데이터 품질 검증